# GPT-2 small: pretraining v2 on a Colab T4

This version uses mixed precision, gradient accumulation, a warm-up/cosine schedule, checkpoints that include optimizer state, and a configurable training-step budget. Keep the same configuration when resuming a run.

In [ ]:
### New
from google.colab import drive
drive.mount('/content/drive')

!pip install -q datasets tiktoken
###New


In [ ]:
### New
import math
import os
from pathlib import Path
import sys

import torch
import torch.nn.functional as F
import tiktoken

for candidate in (Path.cwd(), *Path.cwd().parents):
    module_dir = candidate if (candidate / 'pretraining.py').is_file() else candidate / 'Foundation Model'
    if (module_dir / 'pretraining.py').is_file():
        sys.path.insert(0, str(module_dir))
        break
else:
    raise FileNotFoundError('Could not find Foundation Model/pretraining.py')

from pretraining import (
    GPT_CONFIG_124M,
    create_dataloader_fineweb,
    make_fixed_eval_loaders,
)
from Transformer_arquitectures import GPTModel

if not torch.cuda.is_available():
    raise RuntimeError('This notebook is configured for a CUDA GPU runtime (T4).')

device = torch.device('cuda')
torch.backends.cuda.matmul.allow_tf32 = True
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
###New


In [ ]:
### New
# Do not change these values when resuming from a checkpoint.
MAX_DOCS = None          # None exposes the full streamed FineWeb-Edu corpus.
MAX_LENGTH = 512         # Start here on a T4; use 1024 only after a memory-speed test.
MICRO_BATCH_SIZE = 4     # Reduce to 2 if CUDA runs out of memory.
GRAD_ACCUM_STEPS = 4     # Effective batch = 16 sequences = 8,192 tokens/update.
MAX_TRAIN_STEPS = 20_000 # Optimizer updates in this Colab session; resume for more.
WARMUP_STEPS = 500
LEARNING_RATE = 4e-4
WEIGHT_DECAY = 0.1
VAL_MOD = 100
NUM_WORKERS = 2
EVAL_EVERY = 500
EVAL_BATCHES = 8
SAVE_EVERY = 500
SEED = 123
CHECKPOINT_DIR = Path('/content/drive/MyDrive/llm/checkpoints')
CHECKPOINT_PATH = CHECKPOINT_DIR / 'gpt2_fineweb_v2_latest.pt'
RESUME = True

assert MAX_LENGTH <= 1024
assert MAX_TRAIN_STEPS > WARMUP_STEPS
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Effective tokens/update: {MICRO_BATCH_SIZE * MAX_LENGTH * GRAD_ACCUM_STEPS:,}')
###New


In [ ]:
### New
# num_workers overlaps tokenization/data streaming with GPU work.
train_loader, val_loader = create_dataloader_fineweb(
    batch_size=MICRO_BATCH_SIZE,
    max_length=MAX_LENGTH,
    val_mod=VAL_MOD,
    seed=SEED,
    max_docs=MAX_DOCS,
    num_workers=NUM_WORKERS,
)

# Fixed, small evaluation sets avoid repeatedly traversing the training stream.
train_eval_loader, val_eval_loader = make_fixed_eval_loaders(
    train_loader, val_loader, max_train_batches=EVAL_BATCHES, max_val_batches=EVAL_BATCHES
)

xb, yb = next(iter(train_loader))
print('Batch:', tuple(xb.shape), '| shifted targets:', torch.equal(yb[:, :-1], xb[:, 1:]))
###New


In [ ]:
### New
torch.manual_seed(SEED)
cfg = {**GPT_CONFIG_124M, 'context_length': MAX_LENGTH}
model = GPTModel(cfg).to(device)

# GPT-2 ties token embedding and language-model head weights.
# This changes this implementation from ~162M to ~124M trainable parameters.
model.out_head.weight = model.tok_emb.weight

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY
)

def lr_multiplier(step):
    if step < WARMUP_STEPS:
        return (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / (MAX_TRAIN_STEPS - WARMUP_STEPS)
    return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_multiplier)
scaler = torch.amp.GradScaler('cuda')

n_params = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {n_params:,} ({n_params / 1e6:.1f}M)')
###New


In [ ]:
### New
def evaluate(model, loader, max_batches):
    model.eval()
    losses = []
    with torch.inference_mode():
        for batch_idx, (x, y) in enumerate(loader):
            if batch_idx >= max_batches:
                break
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(x)
                loss = F.cross_entropy(logits.flatten(0, 1), y.flatten())
            losses.append(loss.float().item())
    model.train()
    return sum(losses) / len(losses)

def save_checkpoint(update_step, tokens_seen, history):
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'update_step': update_step,
        'tokens_seen': tokens_seen,
        'history': history,
        'config': {key: value for key, value in cfg.items()},
    }, CHECKPOINT_PATH)

start_update, tokens_seen = 0, 0
history = {'step': [], 'tokens': [], 'train_loss': [], 'val_loss': []}
if RESUME and CHECKPOINT_PATH.exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    if checkpoint['config'] != cfg:
        raise ValueError('Checkpoint configuration differs. Use matching MAX_LENGTH and model configuration.')
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_update = checkpoint['update_step']
    tokens_seen = checkpoint['tokens_seen']
    history = checkpoint['history']
    print(f'Resuming after update {start_update:,}, {tokens_seen / 1e6:.1f}M tokens.')
else:
    print('Starting a new training run.')
###New


In [ ]:
### New
# A resumed streaming run recreates the deterministic stream and skips already-used microbatches.
# Skipping has a startup cost, but prevents training twice on the same initial segment.
microbatches_to_skip = start_update * GRAD_ACCUM_STEPS
train_iter = iter(train_loader)
for _ in range(microbatches_to_skip):
    next(train_iter)

model.train()
optimizer.zero_grad(set_to_none=True)
running_loss = 0.0
for update_step in range(start_update, MAX_TRAIN_STEPS):
    for _ in range(GRAD_ACCUM_STEPS):
        x, y = next(train_iter)
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(x)
            loss = F.cross_entropy(logits.flatten(0, 1), y.flatten())
            scaled_loss = loss / GRAD_ACCUM_STEPS
        scaler.scale(scaled_loss).backward()
        running_loss += loss.detach().float().item()
        tokens_seen += x.numel()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    scheduler.step()

    completed_updates = update_step + 1
    if completed_updates % EVAL_EVERY == 0:
        train_loss = running_loss / EVAL_EVERY / GRAD_ACCUM_STEPS
        val_loss = evaluate(model, val_eval_loader, EVAL_BATCHES)
        history['step'].append(completed_updates)
        history['tokens'].append(tokens_seen)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        print(f'Update {completed_updates:>6,} | tokens {tokens_seen / 1e6:>8.2f}M | '
              f'train {train_loss:.3f} | val {val_loss:.3f} | lr {scheduler.get_last_lr()[0]:.2e}')
        running_loss = 0.0

    if completed_updates % SAVE_EVERY == 0:
        save_checkpoint(completed_updates, tokens_seen, history)
        print(f'Checkpoint saved: {CHECKPOINT_PATH}')

save_checkpoint(MAX_TRAIN_STEPS, tokens_seen, history)
print('Session complete. Run this cell again to resume after increasing MAX_TRAIN_STEPS.')
###New


In [ ]:
### New
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.plot(history['tokens'], history['train_loss'], label='Train loss')
plt.plot(history['tokens'], history['val_loss'], label='Validation loss')
plt.xlabel('Tokens seen')
plt.ylabel('Cross-entropy loss')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

print(f'Total tokens seen: {tokens_seen:,} ({tokens_seen / 1e6:.2f}M)')
print(f'Checkpoint: {CHECKPOINT_PATH}')
###New
